# NeuroDiffusion: A100 Colab Training Launchpad

This notebook handles the end-to-end pipeline: MAE Pre-training -> LDM Fine-tuning -> Image Generation.
**Persistence enabled**: All results and checkpoints are automatically saved to your Google Drive.

### Pre-requisites:
1. Ensure you have `v1-5-pruned.ckpt`, `eeg_5_95_std.pth`, and `block_splits_by_image_single.pth` uploaded directly to your project folder in Google Drive.

## 1. Mount Google Drive

In [43]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Setup Repository

In [44]:
%cd /content
!git clone https://github.com/JoaoLucasVeras/NeuroDiffusion.git
%cd NeuroDiffusion
!git checkout hpc-dev
!git pull

/content
fatal: destination path 'NeuroDiffusion' already exists and is not an empty directory.
/content/NeuroDiffusion
M	code/config.py
M	code/dataset.py
M	code/dc_ldm/ldm_for_eeg.py
M	code/dc_ldm/models/diffusion/ddpm.py
M	code/eval_metrics.py
Already on 'hpc-dev'
Your branch is up to date with 'origin/hpc-dev'.
Already up to date.


## 3. Link Datasets & Persistence
We link your project's output folders to your specific Google Drive path so checkpoints never disappear.

In [45]:
import os
import torch
import numpy as np

# --- CONFIGURATION: Update this path if you move your folder ---
DRIVE_ROOT = '/content/drive/MyDrive/MSAI SPRING 2026/AI ML Project/NeuroDiffusion_Data'

# Ensure required output folders exist on Drive
os.makedirs(f"{DRIVE_ROOT}/results", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/exps", exist_ok=True)

# Create local folder structure in Colab
!mkdir -p datasets
!mkdir -p pretrains/models

# 1. Link Input Data (Files stored directly in NeuroDiffusion_Data)
print("🔗 Linking input files...")
!ln -sf "{DRIVE_ROOT}/eeg_5_95_std.pth" datasets/eeg_5_95_std.pth
!ln -sf "{DRIVE_ROOT}/block_splits_by_image_single.pth" datasets/block_splits_by_image_single.pth
!ln -sf "{DRIVE_ROOT}/v1-5-pruned.ckpt" pretrains/models/v1-5-pruned.ckpt

# 2. Link Output Media (Persistence)
print("🔗 Enabling Persistence...")
!rm -rf results exps # Remove existing empty folders before symlinking
!ln -s "{DRIVE_ROOT}/results" results
!ln -s "{DRIVE_ROOT}/exps" exps

print(f"✅ Linked to: {DRIVE_ROOT}")

# 3. Auto-Unpack for Stage 1
if os.path.exists('datasets/eeg_5_95_std.pth'):
    print("🔨 Generating fragments for Stage 1...")
    os.makedirs('datasets/mne_data', exist_ok=True)
    loaded = torch.load('datasets/eeg_5_95_std.pth')
    for i, item in enumerate(loaded['dataset']):
        np.save(f'datasets/mne_data/sub001_chunk_{i:03d}.npy', item['eeg'].numpy())
    print("✅ Everything is ready.")
else:
    print("❌ ERROR: File not found: {DRIVE_ROOT}/eeg_5_95_std.pth")
    print("Please verify your file name and DRIVE_ROOT path.")

🔗 Linking input files...
🔗 Enabling Persistence...
✅ Linked to: /content/drive/MyDrive/MSAI SPRING 2026/AI ML Project/NeuroDiffusion_Data
🔨 Generating fragments for Stage 1...
✅ Everything is ready.


## 4. Install Dependencies

In [46]:
!pip install einops omegaconf kornia torch-fidelity timm wandb torchmetrics natsort h5py mne transformers pytorch-lightning

## 5. Stage 1: MAE Pre-training

In [47]:
%cd /content/NeuroDiffusion/code
!python stageA1_eeg_pretrain.py --batch_size 128 --num_epoch 500

/content/NeuroDiffusion/code
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/usr/local/lib/python3.12/dist-packages/torchvision/ops/__init__.py", line 1, in <module>
    from ._register_onnx_ops import _register_custom_op
  File "/usr/local/lib/python3.12/dist-packages/torchvision/ops/_register_onnx_ops.py", line 5, in <module>
    from torch.onnx import symbolic_opset11 as opset11
  File "/usr/local/lib/python3.12/dist-packages/torch/onnx/__init__.py", line 27, in <module>
    from . import errors, ops
  File "/usr/local/lib/python3.12/dist-packages/torch/onnx/ops/__init__.py", line 23, in <module>
    from torch.onnx.ops impor

## 6. Stage 2: Diffusion Fine-tuning

In [48]:
import glob
ckpts = sorted(glob.glob('/content/NeuroDiffusion/results/eeg_pretrain/*/checkpoints/checkpoint.pth'))
if not ckpts:
    print("❌ ERROR: Stage 1 checkpoint not found.")
else:
    MAE_CHECKPOINT = ckpts[-1]
    print(f"⭐ Loading latest Stage 1: {MAE_CHECKPOINT}")
    %cd /content/NeuroDiffusion/code
    !git checkout -- eeg_ldm.py dataset.py dc_ldm/models/diffusion/ddpm.py dc_ldm/ldm_for_eeg.py config.py
    !sed -i 's/i <= len(self.dataset.data)/i < len(self.dataset.data)/g' dataset.py
    !sed -i 's/self.images\[self.data\[i\]\["image"\]\]/self.data[i]["image"]/g' dataset.py
    !sed -i "s/map_location='cpu'/map_location='cpu', weights_only=False/g" eeg_ldm.py
    !sed -i "s/weights_only=False, weights_only=False/weights_only=False/g" eeg_ldm.py
    !sed -i 's/map_location="cpu"/map_location="cpu", weights_only=False/g' dc_ldm/ldm_for_eeg.py
    !sed -i 's/from pytorch_lightning.utilities.distributed import rank_zero_only/from pytorch_lightning.utilities.rank_zero import rank_zero_only/g' dc_ldm/models/diffusion/ddpm.py
    # Fix for pytorch-lightning 2.x compatibility: add dataloader_idx to on_train_batch_start
    !sed -i 's/def on_train_batch_start(self, batch, batch_idx):/def on_train_batch_start(self, batch, batch_idx, dataloader_idx=0):/g' dc_ldm/models/diffusion/ddpm.py
    !sed -i 's/def on_train_batch_start(self, batch, batch_idx):/def on_train_batch_start(self, batch, batch_idx, dataloader_idx=0):/g' dc_ldm/ldm_for_eeg.py
    !sed -i 's/subject = 4/subject = 1/g' config.py
    !sed -i 's/subject=4/subject=1/g' config.py
    !python eeg_ldm.py --pretrain_mbm_path {MAE_CHECKPOINT} --batch_size 16

Streaming output truncated to the last 5000 lines.
PLMS Sampler:  68% 170/250 [00:25<00:11,  6.81it/s]
PLMS Sampler:  68% 171/250 [00:25<00:11,  6.81it/s]
PLMS Sampler:  69% 172/250 [00:25<00:11,  6.81it/s]
PLMS Sampler:  69% 173/250 [00:25<00:11,  6.81it/s]
PLMS Sampler:  70% 174/250 [00:25<00:11,  6.80it/s]
PLMS Sampler:  70% 175/250 [00:25<00:11,  6.80it/s]
PLMS Sampler:  70% 176/250 [00:25<00:10,  6.79it/s]
PLMS Sampler:  71% 177/250 [00:26<00:10,  6.80it/s]
PLMS Sampler:  71% 178/250 [00:26<00:10,  6.80it/s]
PLMS Sampler:  72% 179/250 [00:26<00:10,  6.81it/s]
PLMS Sampler:  72% 180/250 [00:26<00:10,  6.80it/s]
PLMS Sampler:  72% 181/250 [00:26<00:10,  6.80it/s]
PLMS Sampler:  73% 182/250 [00:26<00:09,  6.80it/s]
PLMS Sampler:  73% 183/250 [00:26<00:09,  6.81it/s]
PLMS Sampler:  74% 184/250 [00:27<00:09,  6.81it/s]
PLMS Sampler:  74% 185/250 [00:27<00:09,  6.81it/s]
PLMS Sampler:  74% 186/250 [00:27<00:09,  6.81it/s]
PLMS Sampler:  75% 187/250 [00:27<00:09,  6.82it/s]
PLMS Sampler:

## 7. Stage 3: Image Generation

In [49]:
import glob
ckpts = sorted(glob.glob('/content/NeuroDiffusion/exps/eeg_ldm/*/checkpoints/*.ckpt'))
if not ckpts:
    print("❌ ERROR: Stage 2 checkpoint not found.")
else:
    STAGE2_CHECKPOINT = ckpts[-1]
    print(f"⭐ Generating using: {STAGE2_CHECKPOINT}")
    %cd /content/NeuroDiffusion/code
    !python gen_eval_eeg.py --dataset EEG --model_path {STAGE2_CHECKPOINT}

❌ ERROR: Stage 2 checkpoint not found.
